In [1]:
import pandas as pd
import numpy as np
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data = load_all_data()
pl   = data['profitandloss']

print(f"P&L rows: {len(pl)}")
print(f"Companies: {pl['company_id'].nunique()}")
print(f"Year range: {pl['year'].min()} to {pl['year'].max()}")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
P&L rows: 1164
Companies: 100
Year range: 2011-03 to 2024-09


In [2]:
def compute_cagr(start_value, end_value, n_years):
    """
    Computes CAGR with full edge case handling.

    Args:
        start_value: Value at start of period
        end_value:   Value at end of period
        n_years:     Number of years in period

    Returns:
        tuple: (cagr_value or None, flag string)
    """
    # Check for missing values
    if pd.isna(start_value) or pd.isna(end_value):
        return None, 'INSUFFICIENT'

    # Check for insufficient history
    if n_years < 3:
        return None, 'INSUFFICIENT'

    # Check for zero base
    if start_value == 0:
        return None, 'ZERO_BASE'

    # Check for turnaround (negative to positive)
    if start_value < 0 and end_value > 0:
        return None, 'TURNAROUND'

    # Check for decline to loss (positive to negative)
    if start_value > 0 and end_value < 0:
        return None, 'DECLINE_TO_LOSS'

    # Check for both negative
    if start_value < 0 and end_value < 0:
        return None, 'BOTH_NEGATIVE'

    # Normal CAGR computation
    cagr = ((end_value / start_value) ** (1 / n_years) - 1) * 100
    return round(cagr, 2), 'OK'

# Test it
test_cases = [
    (94648, 240893, 9),   # TCS revenue — normal
    (-100,  200,    5),   # Turnaround
    (100,  -200,    5),   # Decline to loss
    (-100, -50,     5),   # Both negative
    (0,     200,    5),   # Zero base
    (100,   150,    2),   # Insufficient history
]

print("CAGR Function Tests:")
for start, end, n in test_cases:
    result, flag = compute_cagr(start, end, n)
    print(f"  ({start:>6}, {end:>6}, {n}yr) → {result} [{flag}]")

CAGR Function Tests:
  ( 94648, 240893, 9yr) → 10.94 [OK]
  (  -100,    200, 5yr) → None [TURNAROUND]
  (   100,   -200, 5yr) → None [DECLINE_TO_LOSS]
  (  -100,    -50, 5yr) → None [BOTH_NEGATIVE]
  (     0,    200, 5yr) → None [ZERO_BASE]
  (   100,    150, 2yr) → None [INSUFFICIENT]


In [3]:
# Let's manually compute TCS revenue CAGR step by step
tcs = pl[pl['company_id'] == 'TCS'].sort_values('year')

print("TCS Revenue History:")
print(tcs[['year', 'sales', 'net_profit', 'eps']].to_string(index=False))

print()

# 5 year CAGR — take row from 5 years ago vs latest
tcs_latest  = tcs.iloc[-1]
tcs_5yr_ago = tcs.iloc[-6] if len(tcs) >= 6 else None

if tcs_5yr_ago is not None:
    cagr_5yr, flag = compute_cagr(
        tcs_5yr_ago['sales'],
        tcs_latest['sales'],
        5
    )
    print(f"TCS Revenue CAGR (5yr): {cagr_5yr}% [{flag}]")
    print(f"  From {tcs_5yr_ago['year']}: ₹{tcs_5yr_ago['sales']:,.0f} Cr")
    print(f"  To   {tcs_latest['year']}: ₹{tcs_latest['sales']:,.0f} Cr")

TCS Revenue History:
   year  sales  net_profit   eps
2013-03  62989       14076  36.0
2014-03  81809       19332  49.0
2015-03  94648       20060  51.0
2016-03 108646       24338  62.0
2017-03 117966       26357  67.0
2018-03 123104       25880  67.0
2019-03 146463       31562  84.0
2020-03 156949       32447  86.0
2021-03 164177       32562  88.0
2022-03 191754       38449 105.0
2023-03 225458       42303 115.0
2024-03 240893       46099 127.0

TCS Revenue CAGR (5yr): 10.46% [OK]
  From 2019-03: ₹146,463 Cr
  To   2024-03: ₹240,893 Cr


In [4]:
def compute_cagr_for_all(pl, metric, windows=[3, 5, 10]):
    """
    Computes CAGR for a given metric across all companies
    for multiple time windows (3yr, 5yr, 10yr).

    Args:
        pl:      Clean P&L DataFrame
        metric:  Column name to compute CAGR for
        windows: List of year windows to compute

    Returns:
        DataFrame: company_id + CAGR columns for each window
    """
    results = []

    for company_id, group in pl.groupby('company_id'):
        group = group.sort_values('year').reset_index(drop=True)
        row   = {'company_id': company_id}

        for n in windows:
            col_name = f'{metric}_cagr_{n}yr'
            flag_name = f'{metric}_cagr_{n}yr_flag'

            if len(group) >= n + 1:
                end_val   = group.iloc[-1][metric]
                start_val = group.iloc[-(n + 1)][metric]
                cagr, flag = compute_cagr(start_val, end_val, n)
            else:
                cagr, flag = None, 'INSUFFICIENT'

            row[col_name]  = cagr
            row[flag_name] = flag

        results.append(row)

    return pd.DataFrame(results)

# Compute for Revenue (sales)
print("Computing Revenue CAGR for all companies...")
revenue_cagr = compute_cagr_for_all(pl, 'sales')
print(f"Done! Shape: {revenue_cagr.shape}")
print(revenue_cagr.head(5).to_string(index=False))

Computing Revenue CAGR for all companies...
Done! Shape: (100, 7)
company_id  sales_cagr_3yr sales_cagr_3yr_flag  sales_cagr_5yr sales_cagr_5yr_flag  sales_cagr_10yr sales_cagr_10yr_flag
       ABB           10.71                  OK            9.72                  OK             9.90                   OK
ADANIENSOL           18.72                  OK           17.85                  OK              NaN            ZERO_BASE
  ADANIENT           34.60                  OK           19.02                  OK             5.78                   OK
ADANIGREEN           43.44                  OK           34.98                  OK              NaN         INSUFFICIENT
ADANIPORTS           28.63                  OK           19.58                  OK            18.65                   OK


In [5]:
# Compute for Net Profit (PAT)
print("Computing PAT CAGR...")
pat_cagr = compute_cagr_for_all(pl, 'net_profit')

# Compute for EPS
print("Computing EPS CAGR...")
eps_cagr = compute_cagr_for_all(pl, 'eps')

# Merge all CAGR results together
cagr_df = revenue_cagr.merge(pat_cagr, on='company_id')
cagr_df = cagr_df.merge(eps_cagr, on='company_id')

print(f"\nFinal CAGR DataFrame shape: {cagr_df.shape}")
print(f"Columns: {cagr_df.columns.tolist()}")

Computing PAT CAGR...
Computing EPS CAGR...

Final CAGR DataFrame shape: (100, 19)
Columns: ['company_id', 'sales_cagr_3yr', 'sales_cagr_3yr_flag', 'sales_cagr_5yr', 'sales_cagr_5yr_flag', 'sales_cagr_10yr', 'sales_cagr_10yr_flag', 'net_profit_cagr_3yr', 'net_profit_cagr_3yr_flag', 'net_profit_cagr_5yr', 'net_profit_cagr_5yr_flag', 'net_profit_cagr_10yr', 'net_profit_cagr_10yr_flag', 'eps_cagr_3yr', 'eps_cagr_3yr_flag', 'eps_cagr_5yr', 'eps_cagr_5yr_flag', 'eps_cagr_10yr', 'eps_cagr_10yr_flag']


In [6]:
# Show CAGR summary for key companies
companies_to_check = ['TCS', 'RELIANCE', 'HDFCBANK', 'INFY', 'MARUTI']

print("Revenue CAGR Summary (5yr):")
print(f"  {'Company':<15} {'Rev CAGR 5yr':>13} {'PAT CAGR 5yr':>13} {'EPS CAGR 5yr':>13}")
print(f"  {'-'*55}")

for company in companies_to_check:
    row = cagr_df[cagr_df['company_id'] == company]
    if len(row) > 0:
        rev  = row['sales_cagr_5yr'].values[0]
        pat  = row['net_profit_cagr_5yr'].values[0]
        eps  = row['eps_cagr_5yr'].values[0]
        rev_str = f"{rev:.1f}%" if rev else "N/A"
        pat_str = f"{pat:.1f}%" if pat else "N/A"
        eps_str = f"{eps:.1f}%" if eps else "N/A"
        print(f"  {company:<15} {rev_str:>13} {pat_str:>13} {eps_str:>13}")

print()
print("Flag Summary (5yr Revenue CAGR):")
print(cagr_df['sales_cagr_5yr_flag'].value_counts().to_string())

Revenue CAGR Summary (5yr):
  Company          Rev CAGR 5yr  PAT CAGR 5yr  EPS CAGR 5yr
  -------------------------------------------------------
  TCS                     10.5%          7.9%          8.6%
  RELIANCE                 9.6%         14.7%         11.9%
  HDFCBANK                21.9%         23.9%         15.4%
  INFY                    13.2%         11.2%         12.5%
  MARUTI                  10.5%         12.0%         11.1%

Flag Summary (5yr Revenue CAGR):
sales_cagr_5yr_flag
OK              99
INSUFFICIENT     1


In [7]:
print("CAGR Coverage Summary:")
print(f"  Total companies: {len(cagr_df)}")
print()

for metric in ['sales', 'net_profit', 'eps']:
    for window in [3, 5, 10]:
        col = f'{metric}_cagr_{window}yr'
        computed = cagr_df[col].notna().sum()
        print(f"  {col:<25} → {computed:>3} companies computed")
    print()

CAGR Coverage Summary:
  Total companies: 100

  sales_cagr_3yr            →  99 companies computed
  sales_cagr_5yr            →  99 companies computed
  sales_cagr_10yr           →  92 companies computed

  net_profit_cagr_3yr       →  91 companies computed
  net_profit_cagr_5yr       →  90 companies computed
  net_profit_cagr_10yr      →  86 companies computed

  eps_cagr_3yr              →  92 companies computed
  eps_cagr_5yr              →  90 companies computed
  eps_cagr_10yr             →  85 companies computed

